Hybrid Retriever- Dense and Sparse matrix

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever


In [2]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

In [3]:
#Dense Retriever (Faiss + huggingface)
embedding_model = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4170.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
#sparse retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs, k=2)

#Combining with dense retriever

hybrid_retriever = EnsembleRetriever(
    retrievers= [
        dense_retriever, sparse_retriever
    ],
    weights = [0.7, 0.3]
)



In [5]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027C43569E80>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000027C4356A660>, k=2)], weights=[0.7, 0.3])

In [6]:
#Query and get results
query = "How can i build an application using LLMs"
results = hybrid_retriever.invoke(query)
results

[Document(id='3e6e07be-4f01-4f43-8282-c0bf98253c04', metadata={}, page_content='LangChain helps build LLM applications.'),
 Document(id='4b51bc6b-8422-445b-8fe1-b951a711eaea', metadata={}, page_content='Langchain can be used to develop agentic ai application.'),
 Document(id='fa0a7a0f-8a52-4769-b1cc-8c402f20c974', metadata={}, page_content='Pinecone is a vector database for semantic search.'),
 Document(id='931047e0-843b-40a3-b4f8-198ed41ae5ef', metadata={}, page_content='Langchain has many types of retrievers.')]

In [7]:
# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Pinecone is a vector database for semantic search.

🔹 Document 4:
Langchain has many types of retrievers.


RAG PIPELINE WITH HYBRID RETRIEVER

In [8]:
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [19]:
#Prompt Template

prompt = PromptTemplate.from_template(
    """ Answer the question based on the following retrieved documents:
    "context" : {context}
     "question": {input}"""
)

In [20]:
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.7)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027C46D09F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027C46D0A490>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [21]:
document_chain = create_stuff_documents_chain(
    llm = llm,
    prompt= prompt
)

In [22]:
retriever_chain = create_retrieval_chain(
    retriever= hybrid_retriever,
    combine_docs_chain= document_chain
)

In [23]:
#Abk the rag chain
query = {"input": "How can I build an app using LLMs?"}
retrieval_response = retriever_chain.invoke(query)
print(retrieval_response)

{'input': 'How can I build an app using LLMs?', 'context': [Document(id='3e6e07be-4f01-4f43-8282-c0bf98253c04', metadata={}, page_content='LangChain helps build LLM applications.'), Document(id='4b51bc6b-8422-445b-8fe1-b951a711eaea', metadata={}, page_content='Langchain can be used to develop agentic ai application.'), Document(id='fa0a7a0f-8a52-4769-b1cc-8c402f20c974', metadata={}, page_content='Pinecone is a vector database for semantic search.'), Document(id='931047e0-843b-40a3-b4f8-198ed41ae5ef', metadata={}, page_content='Langchain has many types of retrievers.')], 'answer': 'Based on the provided documents, to build an app using LLMs (Large Language Models), you can use LangChain. LangChain is designed to help build LLM applications, making it a suitable choice for developing such apps.'}


In [26]:

print("✅ Answer:\n", retrieval_response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(retrieval_response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 Based on the provided documents, to build an app using LLMs (Large Language Models), you can use LangChain. LangChain is designed to help build LLM applications, making it a suitable choice for developing such apps.

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application.

Doc 3: Pinecone is a vector database for semantic search.

Doc 4: Langchain has many types of retrievers.
